# Gold Layer — Agregaciones, KPIs y Métricas de Negocio

**Proyecto 3 — Arquitectura Lakehouse en Azure Databricks**

Este notebook genera las tablas analíticas de la capa Gold a partir de Silver.

**Características:**
- Tablas en formato **Delta**
- Escritura con **MERGE** (reglas de negocio y agregaciones)
- Validación de registros nuevos vs actualizados
- Columna `load_date` en cada tabla

**Tablas Gold:**
| Tabla | Descripción |
|---|---|
| `post_counts_by_user` | Preguntas y respuestas por usuario |
| `vote_stats_per_post` | Votos positivos/negativos por post |
| `top_tags` | Ranking de etiquetas por número de preguntas |
| `user_engagement` | Interacciones totales por usuario |
| `badges_summary` | Insignias por usuario y clase |
| `monthly_activity` | Actividad mensual del período |
| `answer_quality` | Tasa de respuesta aceptada por usuario |

In [0]:
# ============================================================
# CELDA 1 — Parámetros de configuración Gold
# ============================================================
CATALOG        = "lacm_uao_prod_central_us"
BRONZE_SCHEMA  = "bronze"
GOLD_SCHEMA    = "gold"

BRONZE_VOLUME  = "raw_data"
GOLD_VOLUME    = "gold_data"

# Rutas via Unity Catalog Volume (sin account key)
BRONZE_PATH    = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{BRONZE_VOLUME}"
GOLD_PATH      = f"/Volumes/{CATALOG}/{GOLD_SCHEMA}/{GOLD_VOLUME}"

STORAGE_ACCOUNT = "stuaoprod001lacm"
CONTAINER_GOLD  = "gold"
ADLS_GOLD       = f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

YEAR   = 2023
MONTHS = [1, 2]

print(f"[CONFIG] Catálogo:     {CATALOG}")
print(f"[CONFIG] Bronze UC:    {CATALOG}.{BRONZE_SCHEMA}")
print(f"[CONFIG] Gold UC:      {CATALOG}.{GOLD_SCHEMA}")
print(f"[CONFIG] Bronze path:  {BRONZE_PATH}")
print(f"[CONFIG] Gold path:    {GOLD_PATH}")
print(f"[CONFIG] Período:      {MONTHS[0]:02d}/{YEAR} — {MONTHS[-1]:02d}/{YEAR}")


[CONFIG] Catálogo:     lacm_uao_prod_central_us
[CONFIG] Bronze UC:    lacm_uao_prod_central_us.bronze
[CONFIG] Gold UC:      lacm_uao_prod_central_us.gold
[CONFIG] Bronze path:  /Volumes/lacm_uao_prod_central_us/bronze/raw_data
[CONFIG] Gold path:    /Volumes/lacm_uao_prod_central_us/gold/gold_data
[CONFIG] Período:      01/2023 — 02/2023


In [0]:
# ============================================================
# CELDA 2 — Setup: imports, Spark, esquema y volumen Gold
# ⚠ Ejecutar siempre antes de las celdas de agregación
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, max as spark_max,
    countDistinct, when, lit, round as spark_round,
    current_date, explode, split, trim,
    dense_rank, year, month, concat, lpad
)
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, StringType, DoubleType
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
print(f"[OK] Esquema {CATALOG}.{GOLD_SCHEMA} listo.")

# Crear volumen Gold (managed o externo según disponibilidad)
try:
    spark.sql(f"""
        CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.{GOLD_VOLUME}
        LOCATION '{ADLS_GOLD}'
    """)
    print(f"[OK] Volumen externo Gold: {CATALOG}.{GOLD_SCHEMA}.{GOLD_VOLUME}")
except Exception as e:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.{GOLD_VOLUME}")
    print(f"[OK] Volumen managed Gold: {CATALOG}.{GOLD_SCHEMA}.{GOLD_VOLUME}")

print(f"[OK] Ruta Gold: {GOLD_PATH}")


# ── Función MERGE Gold ────────────────────────────────────────
gold_metrics = {}

def merge_to_gold(df_source, table_name: str, merge_key: str, update_cols=None) -> dict:
    """
    MERGE sobre tabla Delta Gold en Unity Catalog.
    Si no existe la crea; si existe hace MERGE por merge_key.
    Usa spark.catalog.tableExists() — compatible con UC.
    Retorna {inserted, updated, total_source}.
    """
    table_full   = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    total_source = df_source.count()

    # Verificar existencia de la tabla en UC (sin JVM privado)
    table_exists = spark.catalog.tableExists(table_full)

    if not table_exists:
        df_source.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_full)
        print(f"  [CREATED] {table_full} — {total_source:,} registros")
        metrics = {"inserted": total_source, "updated": 0, "total_source": total_source}
    else:
        delta_target  = DeltaTable.forName(spark, table_full)
        existing_keys = delta_target.toDF().select(merge_key)
        matched_count = df_source.select(merge_key).join(existing_keys, merge_key, "inner").count()
        insert_count  = total_source - matched_count

        update_set = (
            {c: f"source.{c}" for c in df_source.columns if c != merge_key}
            if update_cols is None
            else {c: f"source.{c}" for c in update_cols}
        )

        delta_target.alias("target") \
            .merge(df_source.alias("source"),
                   f"target.{merge_key} = source.{merge_key}") \
            .whenMatchedUpdate(set=update_set) \
            .whenNotMatchedInsertAll() \
            .execute()

        print(f"  [MERGE] {table_full}: {insert_count:,} insertados | {matched_count:,} actualizados")
        metrics = {"inserted": insert_count, "updated": matched_count, "total_source": total_source}

    gold_metrics[table_name] = metrics
    return metrics


print("[OK] merge_to_gold definida — setup completo.")


[OK] Esquema lacm_uao_prod_central_us.gold listo.
[OK] Volumen managed Gold: lacm_uao_prod_central_us.gold.gold_data
[OK] Ruta Gold: /Volumes/lacm_uao_prod_central_us/gold/gold_data
[OK] merge_to_gold definida — setup completo.


In [0]:
# ============================================================
# CELDA 3 — Leer tablas desde Bronze (Delta en Unity Catalog)
#
# Lee directamente desde lacm_uao_prod_central_us.bronze.*
# Las tablas fueron registradas como Delta por bronze_ingest.
# Se derivan aquí las columnas calculadas que normalmente
# vendrían de Silver: is_answered, post_type_desc, is_positive,
# is_negative, class_desc (no hay capa Silver en este pipeline).
# ============================================================
print("[BRONZE] Cargando tablas desde Unity Catalog...")

df_posts_raw = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.posts")
df_votes_raw = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.votes")
df_comments  = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.comments")
df_users     = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.users")
df_badges_raw= spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.badges")

# ── Columnas derivadas en posts (equivalentes a transformaciones Silver) ──
df_posts = df_posts_raw \
    .withColumn("post_type_desc",
        when(col("PostTypeId") == 1, "Question")
        .when(col("PostTypeId") == 2, "Answer")
        .when(col("PostTypeId") == 5, "Wiki")
        .otherwise("Other")
    ) \
    .withColumn("is_answered",
        when(col("AcceptedAnswerId").isNotNull(), True).otherwise(False)
    ) \
    .withColumn("Tags",
        # Limpiar los símbolos < > que vienen en el campo Tags de Bronze
        trim(col("Tags").cast(StringType()))
    )

# ── Columnas derivadas en votes ───────────────────────────────────────────
df_votes = df_votes_raw \
    .withColumn("is_positive", when(col("VoteTypeId") == 2, True).otherwise(False)) \
    .withColumn("is_negative", when(col("VoteTypeId") == 3, True).otherwise(False))

# ── Columnas derivadas en badges ─────────────────────────────────────────
df_badges = df_badges_raw \
    .withColumn("class_desc",
        when(col("Class") == 1, "Gold")
        .when(col("Class") == 2, "Silver")
        .when(col("Class") == 3, "Bronze")
        .otherwise("Unknown")
    )

# Cache de tablas pequeñas usadas en múltiples joins

print(f"  posts:    {df_posts.count():,} filas")
print(f"  votes:    {df_votes.count():,} filas")
print(f"  comments: {df_comments.count():,} filas")
print(f"  users:    {df_users.count():,} filas")
print(f"  badges:   {df_badges.count():,} filas")
print("[OK] Tablas Bronze listas con columnas derivadas.")


In [0]:
# ============================================================
# CELDA 5 — GOLD 1: post_counts_by_user
# Preguntas y respuestas publicadas por usuario
# ============================================================
print("[GOLD 1/7] Generando post_counts_by_user...")

df_post_counts = df_posts \
    .groupBy("OwnerUserId") \
    .agg(
        count("Id").alias("total_posts"),
        spark_sum(when(col("PostTypeId") == 1, 1).otherwise(0)).alias("total_questions"),
        spark_sum(when(col("PostTypeId") == 2, 1).otherwise(0)).alias("total_answers"),
        avg("Score").alias("avg_score"),
        spark_max("Score").alias("max_score"),
        spark_sum("ViewCount").alias("total_views"),
        spark_sum("FavoriteCount").alias("total_favorites"),
        spark_sum(when(col("is_answered") == True, 1).otherwise(0)).alias("accepted_answers")
    ) \
    .filter(col("OwnerUserId").isNotNull()) \
    .join(
        df_users.select(
            col("Id").alias("OwnerUserId"),
            col("DisplayName"),
            col("Reputation")
        ),
        "OwnerUserId",
        "left"
    ) \
    .withColumn("answer_rate",
        when(col("total_questions") > 0,
            spark_round(col("total_answers") / col("total_questions"), 2)
        ).otherwise(lit(0.0))
    ) \
    .withColumn("avg_score", spark_round(col("avg_score"), 2)) \
    .withColumn("load_date", current_date()) \
    .withColumnRenamed("OwnerUserId", "user_id")

merge_to_gold(df_post_counts, "post_counts_by_user", merge_key="user_id")
print(f"[OK] post_counts_by_user")

[GOLD 1/7] Generando post_counts_by_user...
  [CREATED] lacm_uao_prod_central_us.gold.post_counts_by_user — 199,955 registros
[OK] post_counts_by_user


In [0]:
# ============================================================
# CELDA 6 — GOLD 2: vote_stats_per_post
# Votos positivos/negativos por post
# ============================================================
print("[GOLD 2/7] Generando vote_stats_per_post...")

df_vote_stats = df_votes \
    .groupBy("PostId") \
    .agg(
        count("Id").alias("total_votes"),
        spark_sum(when(col("is_positive") == True, 1).otherwise(0)).alias("upvotes"),
        spark_sum(when(col("is_negative") == True, 1).otherwise(0)).alias("downvotes"),
        spark_sum(when(col("VoteTypeId") == 5, 1).otherwise(0)).alias("favorites"),
        spark_sum(when(col("VoteTypeId") == 4, 1).otherwise(0)).alias("offensive_flags"),
        spark_sum("BountyAmount").alias("total_bounty")
    ) \
    .filter(col("PostId").isNotNull()) \
    .join(
        df_posts.select(
            col("Id").alias("PostId"),
            col("Score"),
            col("post_type_desc"),
            col("Title"),
            col("OwnerUserId")
        ),
        "PostId",
        "left"
    ) \
    .withColumn("vote_ratio",
        when((col("upvotes") + col("downvotes")) > 0,
            spark_round(col("upvotes") / (col("upvotes") + col("downvotes")), 3)
        ).otherwise(lit(0.0))
    ) \
    .withColumn("net_votes", col("upvotes") - col("downvotes")) \
    .withColumn("load_date", current_date()) \
    .withColumnRenamed("PostId", "post_id")

merge_to_gold(df_vote_stats, "vote_stats_per_post", merge_key="post_id")
print(f"[OK] vote_stats_per_post")

[GOLD 2/7] Generando vote_stats_per_post...
  [CREATED] lacm_uao_prod_central_us.gold.vote_stats_per_post — 1,421,530 registros
[OK] vote_stats_per_post


In [0]:
# ============================================================
# CELDA 7 — GOLD 3: top_tags
# Ranking de etiquetas con mayor número de preguntas
# ============================================================
print("[GOLD 3/7] Generando top_tags...")

# Explode de tags: cada post puede tener múltiples tags separados por espacio
df_questions = df_posts.filter(col("PostTypeId") == 1)

df_tags_exploded = df_questions \
    .filter(col("Tags").isNotNull() & (col("Tags") != "")) \
    .withColumn("tag", explode(split(trim(col("Tags")), " "))) \
    .withColumn("tag", trim(col("tag"))) \
    .filter(col("tag") != "")

# Window para el ranking

df_top_tags = df_tags_exploded \
    .groupBy("tag") \
    .agg(
        count("Id").alias("question_count"),
        avg("Score").alias("avg_score"),
        spark_sum("ViewCount").alias("total_views"),
        spark_sum("AnswerCount").alias("total_answers"),
        spark_sum("FavoriteCount").alias("total_favorites"),
        countDistinct("OwnerUserId").alias("distinct_users")
    ) \
    .withColumn("avg_score", spark_round(col("avg_score"), 2)) \
    .withColumn("avg_answers_per_question",
        when(col("question_count") > 0,
            spark_round(col("total_answers") / col("question_count"), 2)
        ).otherwise(lit(0.0))
    ) \
    .withColumn("rank",
        dense_rank().over(
            Window.partitionBy(lit(1)).orderBy(col("question_count").desc())
        )
    ) \
    .withColumn("load_date", current_date())

merge_to_gold(df_top_tags, "top_tags", merge_key="tag")
print(f"[OK] top_tags")

In [0]:
# ============================================================
# CELDA 8 — GOLD 4: user_engagement
# Interacciones de cada usuario: posts, comentarios, votos, badges
# ============================================================
print("[GOLD 4/7] Generando user_engagement...")

# Agregaciones por separado
agg_posts = df_posts \
    .filter(col("OwnerUserId").isNotNull()) \
    .groupBy(col("OwnerUserId").alias("user_id")) \
    .agg(
        count("Id").alias("post_count"),
        spark_sum("Score").alias("total_post_score")
    )

agg_comments = df_comments \
    .filter(col("UserId").isNotNull()) \
    .groupBy(col("UserId").alias("user_id")) \
    .agg(
        count("Id").alias("comment_count"),
        spark_sum("Score").alias("total_comment_score")
    )

agg_votes_cast = df_votes \
    .filter(col("UserId").isNotNull()) \
    .groupBy(col("UserId").alias("user_id")) \
    .agg(
        count("Id").alias("votes_cast"),
        spark_sum(when(col("is_positive"), 1).otherwise(0)).alias("upvotes_cast"),
        spark_sum(when(col("is_negative"), 1).otherwise(0)).alias("downvotes_cast")
    )

agg_badges = df_badges \
    .groupBy(col("UserId").alias("user_id")) \
    .agg(
        count("Id").alias("badge_count"),
        spark_sum(when(col("class_desc") == "Gold",   1).otherwise(0)).alias("gold_badges"),
        spark_sum(when(col("class_desc") == "Silver", 1).otherwise(0)).alias("silver_badges"),
        spark_sum(when(col("class_desc") == "Bronze", 1).otherwise(0)).alias("bronze_badges")
    )

# Join de todas las dimensiones sobre users
df_engagement = df_users \
    .select(
        col("Id").alias("user_id"),
        col("DisplayName"),
        col("Reputation"),
        col("CreationDate"),
        col("Location")
    ) \
    .join(agg_posts,      "user_id", "left") \
    .join(agg_comments,   "user_id", "left") \
    .join(agg_votes_cast, "user_id", "left") \
    .join(agg_badges,     "user_id", "left") \
    .fillna(0, subset=[
        "post_count", "total_post_score",
        "comment_count", "total_comment_score",
        "votes_cast", "upvotes_cast", "downvotes_cast",
        "badge_count", "gold_badges", "silver_badges", "bronze_badges"
    ]) \
    .withColumn("total_interactions",
        col("post_count") + col("comment_count") + col("votes_cast")
    ) \
    .withColumn("engagement_score",
        # Fórmula: posts*3 + comentarios*1 + votos*0.5 + badges*2
        spark_round(
            col("post_count")    * 3.0 +
            col("comment_count") * 1.0 +
            col("votes_cast")    * 0.5 +
            col("badge_count")   * 2.0,
            2
        )
    ) \
    .withColumn("load_date", current_date())

merge_to_gold(df_engagement, "user_engagement", merge_key="user_id")
print(f"[OK] user_engagement")

[GOLD 4/7] Generando user_engagement...
  [CREATED] lacm_uao_prod_central_us.gold.user_engagement — 400,042 registros
[OK] user_engagement


In [0]:
# ============================================================
# CELDA 9 — GOLD 5: badges_summary
# Número y tipo de insignias por usuario
# ============================================================
print("[GOLD 5/7] Generando badges_summary...")

df_badges_summary = df_badges \
    .groupBy("UserId", "class_desc") \
    .agg(
        count("Id").alias("badge_count"),
        countDistinct("Name").alias("distinct_badge_names"),
        spark_sum(when(col("TagBased") == 1, 1).otherwise(0)).alias("tag_based_count")
    ) \
    .join(
        df_users.select(
            col("Id").alias("UserId"),
            col("DisplayName"),
            col("Reputation")
        ),
        "UserId",
        "left"
    ) \
    .withColumn("user_id",    col("UserId").cast(LongType())) \
    .withColumn("badge_class", col("class_desc")) \
    .withColumn("summary_key",
        concat(col("UserId").cast(StringType()), lit("_"), col("class_desc"))
    ) \
    .withColumn("load_date", current_date()) \
    .drop("UserId", "class_desc")

merge_to_gold(df_badges_summary, "badges_summary", merge_key="summary_key")
print(f"[OK] badges_summary")

In [0]:
# ============================================================
# CELDA 10 — GOLD 6: monthly_activity
# Actividad mensual del período (posts + votos + comentarios)
# ============================================================
print("[GOLD 6/7] Generando monthly_activity...")

agg_posts_monthly = df_posts \
    .withColumn("yr",  year("CreationDate")) \
    .withColumn("mo",  month("CreationDate")) \
    .groupBy("yr", "mo") \
    .agg(
        count("Id").alias("posts_created"),
        spark_sum(when(col("PostTypeId") == 1, 1).otherwise(0)).alias("questions_created"),
        spark_sum(when(col("PostTypeId") == 2, 1).otherwise(0)).alias("answers_created"),
        avg("Score").alias("avg_post_score"),
        countDistinct("OwnerUserId").alias("active_users_posts")
    )

agg_votes_monthly = df_votes \
    .withColumn("yr",  year("CreationDate")) \
    .withColumn("mo",  month("CreationDate")) \
    .groupBy("yr", "mo") \
    .agg(
        count("Id").alias("votes_cast"),
        spark_sum(when(col("is_positive"), 1).otherwise(0)).alias("upvotes"),
        spark_sum(when(col("is_negative"), 1).otherwise(0)).alias("downvotes")
    )

agg_comments_monthly = df_comments \
    .withColumn("yr",  year("CreationDate")) \
    .withColumn("mo",  month("CreationDate")) \
    .groupBy("yr", "mo") \
    .agg(
        count("Id").alias("comments_created"),
        countDistinct("UserId").alias("active_users_comments")
    )

df_monthly = agg_posts_monthly \
    .join(agg_votes_monthly,    ["yr", "mo"], "outer") \
    .join(agg_comments_monthly, ["yr", "mo"], "outer") \
    .fillna(0) \
    .withColumn("period_key",
        concat(col("yr").cast(StringType()), lit("-"), lpad(col("mo").cast(StringType()), 2, "0"))
    ) \
    .withColumn("avg_post_score", spark_round(col("avg_post_score"), 2)) \
    .withColumn("vote_positivity",
        when((col("upvotes") + col("downvotes")) > 0,
            spark_round(col("upvotes") / (col("upvotes") + col("downvotes")), 3)
        ).otherwise(lit(0.0))
    ) \
    .withColumn("load_date", current_date())

merge_to_gold(df_monthly, "monthly_activity", merge_key="period_key")
print(f"[OK] monthly_activity")

[GOLD 6/7] Generando monthly_activity...
  [CREATED] lacm_uao_prod_central_us.gold.monthly_activity — 2 registros
[OK] monthly_activity


In [0]:
# ============================================================
# CELDA 11 — GOLD 7: answer_quality
# Tasa de aceptación de respuestas por usuario (calidad)
# ============================================================
print("[GOLD 7/7] Generando answer_quality...")

# Solo answers
df_answers = df_posts.filter(col("PostTypeId") == 2)

df_answer_quality = df_answers \
    .filter(col("OwnerUserId").isNotNull()) \
    .groupBy(col("OwnerUserId").alias("user_id")) \
    .agg(
        count("Id").alias("total_answers"),
        spark_sum(when(col("AcceptedAnswerId").isNotNull(), 1).otherwise(0)).alias("accepted_answers"),
        avg("Score").alias("avg_answer_score"),
        spark_max("Score").alias("max_answer_score"),
        spark_sum("Score").alias("total_score")
    ) \
    .join(
        df_users.select(
            col("Id").alias("user_id"),
            col("DisplayName"),
            col("Reputation")
        ),
        "user_id",
        "left"
    ) \
    .withColumn("acceptance_rate",
        when(col("total_answers") > 0,
            spark_round(col("accepted_answers") / col("total_answers"), 3)
        ).otherwise(lit(0.0))
    ) \
    .withColumn("avg_answer_score", spark_round(col("avg_answer_score"), 2)) \
    .withColumn("quality_tier",
        when(col("acceptance_rate") >= 0.5,  "High")
        .when(col("acceptance_rate") >= 0.25, "Medium")
        .otherwise("Low")
    ) \
    .withColumn("load_date", current_date())

merge_to_gold(df_answer_quality, "answer_quality", merge_key="user_id")
print(f"[OK] answer_quality")

[GOLD 7/7] Generando answer_quality...
  [CREATED] lacm_uao_prod_central_us.gold.answer_quality — 96,057 registros
[OK] answer_quality


In [0]:
# ============================================================
# CELDA 12 — Reporte consolidado Gold
# ============================================================
print("\n" + "=" * 70)
print("RESUMEN DE AGREGACIÓN GOLD")
print("=" * 70)
print(f"{'Tabla Gold':<28} | {'Fuente':>8} | {'Insertados':>10} | {'Actualizados':>12}")
print("-" * 70)

total_ins = total_upd = total_src = 0
for tname, m in gold_metrics.items():
    ins = m.get("inserted", 0)
    upd = m.get("updated",  0)
    src = m.get("total_source", 0)
    total_ins += ins
    total_upd += upd
    total_src += src
    print(f"{tname:<28} | {src:>8,} | {ins:>10,} | {upd:>12,}")

print("-" * 70)
print(f"{'TOTAL':<28} | {total_src:>8,} | {total_ins:>10,} | {total_upd:>12,}")
print("\n[OK] Capa Gold completada. Tablas disponibles en Unity Catalog.")


RESUMEN DE AGREGACIÓN GOLD
Tabla Gold                   |   Fuente | Insertados | Actualizados
----------------------------------------------------------------------
post_counts_by_user          |  199,955 |    199,955 |            0
vote_stats_per_post          | 1,421,530 |  1,421,530 |            0
top_tags                     |  121,157 |    121,157 |            0
user_engagement              |  400,042 |    400,042 |            0
badges_summary               |  512,510 |    512,510 |            0
monthly_activity             |        2 |          2 |            0
answer_quality               |   96,057 |     96,057 |            0
----------------------------------------------------------------------
TOTAL                        | 2,751,253 |  2,751,253 |            0

[OK] Capa Gold completada. Tablas disponibles en Unity Catalog.


In [0]:
# ============================================================
# CELDA 13 — Validación de tablas Gold
# ============================================================
print("[CALIDAD] Validando tablas Gold...\n")

gold_tables = [
    ("post_counts_by_user", "user_id"),
    ("vote_stats_per_post", "post_id"),
    ("top_tags",            "tag"),
    ("user_engagement",     "user_id"),
    ("badges_summary",      "summary_key"),
    ("monthly_activity",    "period_key"),
    ("answer_quality",      "user_id"),
]

for tname, key in gold_tables:
    try:
        df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{tname}")
        total = df.count()
        nulls = df.filter(col(key).isNull()).count()
        dups  = total - df.dropDuplicates([key]).count()
        has_load = "load_date" in df.columns
        status = "✓" if nulls == 0 and dups == 0 else "⚠"
        print(f"{status} {tname:<28}: filas={total:,} | nulls_key={nulls} | dups={dups} | load_date={has_load}")
    except Exception as e:
        print(f"✗ {tname}: {e}")


[CALIDAD] Validando tablas Gold...

✓ post_counts_by_user         : filas=199,955 | nulls_key=0 | dups=0 | load_date=True
✓ vote_stats_per_post         : filas=1,421,530 | nulls_key=0 | dups=0 | load_date=True
✓ top_tags                    : filas=121,157 | nulls_key=0 | dups=0 | load_date=True
✓ user_engagement             : filas=400,042 | nulls_key=0 | dups=0 | load_date=True
✓ badges_summary              : filas=512,510 | nulls_key=0 | dups=0 | load_date=True
✓ monthly_activity            : filas=2 | nulls_key=0 | dups=0 | load_date=True
✓ answer_quality              : filas=96,057 | nulls_key=0 | dups=0 | load_date=True


In [0]:
# Instalar duckdb — compatible con Job y ejecución manual
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
print('[OK] duckdb instalado.')

  Obtaining dependency information for duckdb from https://files.pythonhosted.org/packages/dc/a2/67694010693ec8c8c975e6991f48ef886d35ecbdaa2f287234882a403c21/duckdb-1.5.2-cp311-cp311-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/21.4 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/21.4 MB 3.0 MB/s eta 0:00:08
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/21.4 MB 4.8 MB/s eta 0:00:05
   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.6/21.4 MB 5.6 MB/s eta 0:00:04
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.9/21.4 MB 6.4 MB/s eta 0:00:04
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/21.4 MB 7.1 MB/s eta 0:00:03
   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/21.4 MB 7.7 MB/s eta 0:00:03
   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/21.4 MB 8.5 MB/s eta 0:00:03
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/21.4 MB 9.2 MB/s eta 0:00:03
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/21.4 MB 10.0 MB/s eta 0:00

In [0]:
# ============================================================
# CELDA 14 — Consultas DuckDB sobre tablas Gold
# ============================================================
import duckdb

con = duckdb.connect()

# --- Top 10 Tags ---
df_tags_pd = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.top_tags") \
    .orderBy(col("question_count").desc()).limit(200).toPandas()

print("\n[DUCKDB] Top 10 tags por número de preguntas:")
print(con.execute("""
    SELECT tag, question_count, avg_score, rank
    FROM df_tags_pd ORDER BY rank ASC LIMIT 10
""").df().to_string(index=False))

# --- Top 10 usuarios por engagement ---
df_eng_pd = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.user_engagement") \
    .orderBy(col("engagement_score").desc()).limit(200).toPandas()

print("\n[DUCKDB] Top 10 usuarios por engagement_score:")
print(con.execute("""
    SELECT user_id, DisplayName, engagement_score,
           post_count, comment_count, badge_count
    FROM df_eng_pd ORDER BY engagement_score DESC LIMIT 10
""").df().to_string(index=False))

# --- Actividad mensual ---
df_monthly_pd = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.monthly_activity").toPandas()

print("\n[DUCKDB] Resumen de actividad mensual:")
print(con.execute("""
    SELECT period_key, posts_created, questions_created,
           answers_created, votes_cast, comments_created,
           ROUND(vote_positivity * 100, 1) AS positivity_pct
    FROM df_monthly_pd ORDER BY period_key
""").df().to_string(index=False))

print("\n[OK] Consultas DuckDB completadas.")



[DUCKDB] Top 10 tags por número de preguntas:
                      tag  question_count  avg_score  rank
                 |python|            1343      -0.28     1
             |javascript|             941       0.01     2
           |flutter|dart|             745       0.29     3
               |html|css|             716       0.08     4
                      |r|             710       0.43     5
          |python|pandas|             700       0.35     6
|python|pandas|dataframe|             623       0.51     7
              |excel|vba|             544       0.23     8
             |typescript|             538       0.61     9
                |flutter|             535       0.31    10

[DUCKDB] Top 10 usuarios por engagement_score:
 user_id                                                                              DisplayName  engagement_score  post_count  comment_count  badge_count
21021990                                        [77, 105, 99, 104, 97, 101, 108, 32, 67, 97, 111]   

In [0]:
# ============================================================
# CELDA 15 — Instrucciones para conectar con Power BI
# ============================================================
print(f"""
=== CONEXIÓN POWER BI ===

1. En Databricks: Compute → tu cluster → Advanced Options → JDBC/ODBC
   Copiar: Server Hostname y HTTP Path

2. En Power BI Desktop:
   Obtener datos → Azure Databricks
   Server:    <server_hostname>
   HTTP Path: <http_path>
   Catálogo:  {CATALOG}
   Esquema:   {GOLD_SCHEMA}

3. Tablas disponibles ({CATALOG}.{GOLD_SCHEMA}):
   - post_counts_by_user  → preguntas y respuestas por usuario
   - vote_stats_per_post  → votos positivos/negativos por post
   - top_tags             → ranking de tecnologías
   - user_engagement      → score de engagement por usuario
   - badges_summary       → insignias por usuario y clase
   - monthly_activity     → tendencia mensual 01-02/2023
   - answer_quality       → tasa de aceptación por respondedor

4. Visualizaciones sugeridas:
   - Barras horizontales: top 20 tags por question_count
   - Líneas de área:      posts_created + votes_cast por mes
   - Scatter:             engagement_score vs Reputation
   - Dona:                distribución gold/silver/bronze badges
   - Tabla KPI:           top usuarios por acceptance_rate
""")



=== CONEXIÓN POWER BI ===

1. En Databricks: Compute → tu cluster → Advanced Options → JDBC/ODBC
   Copiar: Server Hostname y HTTP Path

2. En Power BI Desktop:
   Obtener datos → Azure Databricks
   Server:    <server_hostname>
   HTTP Path: <http_path>
   Catálogo:  lacm_uao_prod_central_us
   Esquema:   gold

3. Tablas disponibles (lacm_uao_prod_central_us.gold):
   - post_counts_by_user  → preguntas y respuestas por usuario
   - vote_stats_per_post  → votos positivos/negativos por post
   - top_tags             → ranking de tecnologías
   - user_engagement      → score de engagement por usuario
   - badges_summary       → insignias por usuario y clase
   - monthly_activity     → tendencia mensual 01-02/2023
   - answer_quality       → tasa de aceptación por respondedor

4. Visualizaciones sugeridas:
   - Barras horizontales: top 20 tags por question_count
   - Líneas de área:      posts_created + votes_cast por mes
   - Scatter:             engagement_score vs Reputation
   - Dona